# 🚀 Analyse 1: Diachrone Frequenzdiagramme der Spanischen Grippe

Teil der Fallstudie „Quantitative Analyse der Medienwellen der Spanischen Grippe (1918/19)“

## „In früheren Folgen“ (etwas Kontext)

Dieses Notebook ist nur ein Teil einer größeren didaktischen Fallstudie. Die konkrete Fallstudie widmet sich der folgenden Forschungsfrage:

> Lassen sich für die Spanische Grippe 1918/1919 mit Fokus auf den Berliner Raum Muster in der öffentlichen Aufmerksamkeit ausmachen, die eine wellenartige Verlaufsform aufweisen?

Weitere Informationen dazu finden sich im [Unterkapitel „Fragestellung“](../research_question/research-question_research-question.md).

In den vorherigen Teilen dieser Fallstudie haben wir ein Korpus aus zwei Berliner Zeitungen der Jahre 1918 und 1919 zusammengestellt (siehe [„Aufbau des Forschungskorpus“](../corpus_collection/corpus-collection_building-our-corpus.md)). Anschließend haben wir digitale Bilder mithilfe von OCR-Verfahren in Text umgewandelt (siehe [„OCR“](../ocr/ocr.md)). Danach wurden die digitalisierten Texte mit dem NLP-Tool spaCy verarbeitet, um tokenisierte und lemmatisierte Texte zu erhalten (siehe [„Korpusverarbeitung“](../corpus_processing/corpus-processing_intro.md)).

In diesem Notebook setzen wir die Fallstudie mit einer quantitativen Exploration dieser Texte fort. Dabei versuchen wir, die „Medienwellen“ der Spanischen Grippe anhand kombinierter Worthäufigkeiten in verschiedenen Zeitabschnitten sichtbar zu machen.



## Übersicht über dieses Notebook 
Im Folgenden werden die von SpaCy annotierten Dateien (CSV-Format) analysiert. Unser Ziel ist es, die Wort-/Lemma-Häufigkeiten einer vordefinierten Wortgruppe für die Monate des Jahres 1918 zu plotten und zu sehen, ob sie mit den Wellen der Grippepandemie korrelieren.
Dafür werden folgende Schritte durchgeführt:
1. Einlesen des Korpus, der Metadaten und der Grippe-Wortliste
2. Extraktion der Worthäufigkeiten und Plotten der Worthäufigkeiten
3. Diskussion der Ergebnisse

## Bevor die Action beginnt: kurze Einführung in Jupyter-Notebooks

Was Sie im Moment sehen, ist ein **Jupyter-Notebook**. Jupyter ermöglicht es Ihnen, Python-Code in Ihrem Browser zu schreiben und auszuführen. Es gibt Jupyter-Zellen mit Texten (wie diese hier) und Zellen mit Code. Notebook-Zellen mit Code sehen so aus:

In [ ]:
print('Hallo Welt!')

Um **Code in der Zelle auszuführen**, treten Sie darauf und drücken Sie **Cmd/Strg + Enter** oder verwenden Sie die **'Play button'** links neben der Zelle.


Eine weitere nützliche Tastenkombination ist **Umschalt+Enter**. Sie führt die Shell aus und geht zur nächsten über. Nützlich, wenn Sie durch viele Zellen klicken müssen. 

<details>
  <summary><b>Informationen zum Ausführen des Notebooks – Zum Ausklappen klicken ⬇️</b></summary>
  
<b>Voraussetzungen zur Ausführung des Jupyter Notebooks</b>
<ol>
<li> Installieren der Bibliotheken </li>
<li> Pfad zu den Daten setzen</li>
<li> Laden der Daten (z.B. über den Command `wget` (s.u.))</li>
</ol>
Zum Testen: Ausführen der Zelle „load libraries“ und der Sektion „Einlesen der Daten“. </br>
Alle Zellen, die mit 🚀 gekennzeichnet sind, werden nur bei der Ausführung des Notebooks in Colab / JupyterHub bzw. lokal ausgeführt. 
</details>

In [ ]:
#  🚀 Install libraries 
! pip install pandas plotly 

In [ ]:
import re
import requests
from pathlib import Path
import pandas as pd

import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"

## Einlesen der Daten, Metadaten und der Grippe-Wortliste
Um eine/mehrere Dateien mit Python bearbeiten zu können, müssen die Dateien zuerst ausgewählt werden, d. h., der [Pfad](https://en.wikipedia.org/wiki/Path_(computing)) zu den Dateien wird gesetzt und anschließend werden die Dateien eingelesen. 

### Einlesen des Korpus (CSV-Dateien)

<details>
  <summary><b>Informationen zum Ausführen des Notebooks – Zum Ausklappen klicken ⬇️</b></summary>
Zuerst wird der Ordner angelegt, in dem die CSV-Dateien gespeichert werden. Der Einfachheit halber wird die gleiche Datenablagestruktur wie in dem <a href="https://github.com/quadriga-dk/Text-Fallstudie-1/tree/main">GitHub Repository</a>, in dem die Daten gespeichert sind, vorausgesetzt. </br>
Danach werden alle CSV-Dateien im Korpus heruntergeladen und gespeichert. Dafür sind folgende Schritte nötig:
<ol>
    <li>Es wird eine Liste erstellt, die die URLs zu den einzelnen CSV-Dateien beinhaltet.</li>
    <li>Die Liste wird als txt-Datei gespeichert.</li>
    <li>Alle Dateien aus der Liste werden heruntergeladen und in dem Ordner <i>../data/csv</i> gespeichert.</li>
</ol>
Sollten die Dateien schon an einem anderen Ort vorhanden sein, können die Dateipfade zu den Ordnern angepasst werden. </br>
</details>

Setzen des Pfads:

In [ ]:
# set the path to csv files to be processed
csv_dir = Path(r"../data/csv")

Einlesen der CSV-Dateien

In [ ]:
# Create dictionary to save the corpus data (filenames and tables)
corpus_annotations = {}

# Iterate over csv files 
for file in csv_dir.iterdir():
    # check if the entry is a file, not a directory
    if file.is_file():
        # check if the file has the correct suffix csv
        if file.suffix == '.csv':
            # read the csv table to a data frame
            data = pd.read_csv(file) 
            # save the data frame to the dictionary, key=filename (without suffix), value=dataframe
            corpus_annotations[file.with_suffix("").name] = data

Wie viele Dateien wurden eingelesen?

In [ ]:
len(corpus_annotations)

Wie sieht der Anfang der ersten Datei aus?

In [ ]:
corpus_annotations[list(corpus_annotations.keys())[5]].head()

### Einlesen der Metadaten

<details>
  <summary><b>Informationen zum Ausführen des Notebooks – Zum Ausklappen klicken ⬇️</b></summary>
Der Pfad kann in der Variable <i>metadata_path</i> angepasst werden. Die einzulesende Datei muss die Endung `.csv` haben. </br>
</details>

In [ ]:
# set path to metadata file
metadata_path = '../data/metadata/QUADRIGA_FS-Text-01_Data01_Corpus-Table.csv'

# read metadata file to pandas dataframe
corpus_metadata = pd.read_csv(metadata_path, sep=';')
corpus_metadata['DC.date'] = pd.to_datetime(corpus_metadata['DC.date'])
#corpus_metadata = corpus_metadata.set_index('DC.identifier')

Wie sieht die Metadaten-Datei aus? (erste fünf Zeilen)

In [ ]:
corpus_metadata.head()

### Einlesen der Wortliste (Semantisches Feld „Grippe“)

#### Erläuterung: Semantisches Feld
Das Ziel der Analyse ist es, zu quantifizieren, wie viel über die Spanische Grippe berichtet wird. Dafür sollen möglichst alle und nur die Textstellen erfasst werden, in denen die Spanische Grippe erwähnt wird. Eine Erwähnung liegt dann vor, wenn ein Wort vorkommt, das mit der Spanischen Grippe im Zusammenhang steht. Die Sammlung dieser Wörter nennen wir Semantisches Feld. Da die Wörter losgelöst von ihrem Kontext analysiert werden, sollten sie so gewählt sein, dass sie sich auf die Spanische Grippe und nur auf diese beziehen.

<details>
  <summary><b>Informationen zum Ausführen des Notebooks – Zum Ausklappen klicken ⬇️</b></summary>
In der folgenden Codezelle legen wir den Pfad zur Textdatei fest, in der die Liste gespeichert ist. Anschließend lesen wir den Text aus der Datei und teilen ihn in Wörter auf (anhand von Zeilenumbrüchen).`
</details>

In [ ]:
path_to_wordlist = Path("../data/wordlist/grippe.txt")
word_list = path_to_wordlist.read_text().split("\n")

Wie sieht die Wortliste aus?

In [ ]:
word_list

## Suche nach einem Lemma und plotte die Häufigkeit

1. Datum zu den Annotationen hinzufügen
2. Annotationen in einer Datenstruktur (einem DataFrame) speichern
3. Lemmata suchen und nach Zeitabschnitt gruppieren
4. Häufigkeiten plotten

### Datum zu den Annotationen hinzufügen

In [ ]:
def add_date_to_corpus_annotations(corpus_metadata: pd.DataFrame, corpus_annotated: dict[str, pd.DataFrame]) -> None:
    """Add date colum from corpus_metadata to corpus annotated. Map by DC.identifier / filename."""
    for identifier, df in corpus_annotated.items():
        if identifier in corpus_metadata["DC.identifier"].values:
            df["date"] = corpus_metadata[corpus_metadata["DC.identifier"] == identifier]["DC.date"].item()

In [ ]:
add_date_to_corpus_annotations(corpus_metadata, corpus_annotations)

### Annotationen in einer Datenstruktur (einem DataFrame) speichern

In [ ]:
corpus_annotations_merged = pd.concat(corpus_annotations.values())

### Berechnen der absoluten und relativen Häufigkeiten zusammengefasst pro Tag, Woche und Monat

In [ ]:
def search_split_by_timeframe(merged_df: pd.DataFrame, search_terms=word_list) -> tuple[dict[str, pd.DataFrame], dict[str, pd.DataFrame]]:
    """Get lemmata count of words in search_terms by month, week and days.
    :param pd.DataFrame merged_df: The merged dataframe of all annotations
    :param list search_terms: List of words to search in merged_df
    :return tuple: Two dictionaries with identical keys, saving the absolute and relative frequencies by three time frames respectively
    """
    # Filter dataframe by lemmata in word_list
    result = merged_df.query(f'Lemma.isin({search_terms})')

    # Collect lemmata count by time frames: month, week, day
    frequency_parameters = ["M", "W-MON", "D"]
    absolute_frequencies = {option: result.groupby(pd.PeriodIndex(result['date'], freq=option)).count().Lemma 
                            for option in frequency_parameters}

    relative_frequencies = {}
    for frequency_param in frequency_parameters:
        relative_frequencies[frequency_param] = absolute_frequencies[frequency_param] / merged_df.groupby(pd.PeriodIndex(merged_df['date'], freq=frequency_param)).count().Lemma.fillna(0)

    return absolute_frequencies, relative_frequencies

### Erstellen eines interaktiven Liniendiagramms

In [ ]:
def plot_frequencies(merged_df: pd.DataFrame, search_terms=word_list) -> None:
    """
    Plot lemmata frequencies of words in search_terms over time, interactively.
    A Plotly dropdown menu lets the user switch between monthly/weekly/daily
    aggregation and absolute/relative counts (six combinations total).
    :param pd.DataFrame merged_df: The merged dataframe of all annotations
    :param list search_terms: List of words to search in merged_df
    """
    absolute_frequencies, relative_frequencies = search_split_by_timeframe(
        merged_df, search_terms=search_terms
    )

    views = [
        ("Monatlich – Absolut",   absolute_frequencies["M"]),
        ("Monatlich – Relativ",   relative_frequencies["M"]),
        ("Wöchentlich – Absolut", absolute_frequencies["W-MON"]),
        ("Wöchentlich – Relativ", relative_frequencies["W-MON"]),
        ("Täglich – Absolut",     absolute_frequencies["D"]),
        ("Täglich – Relativ",     relative_frequencies["D"]),
    ]

    # PeriodIndex → Timestamp so Plotly renders a datetime x-axis.
    def xy(series):
        return series.index.to_timestamp(), series.values

    default_label, default_series = views[0]
    x0, y0 = xy(default_series)

    fig = go.Figure(
        data=[go.Scatter(x=x0, y=y0, mode="lines", line=dict(width=2))]
    )

    buttons = []
    for label, series in views:
        x, y = xy(series)
        buttons.append(dict(
            label=label,
            method="update",
            args=[
                {"x": [list(x)], "y": [list(y)]},
                {"yaxis": {"title": "Relative Häufigkeit" if "Relativ" in label
                                                          else "Absolute Häufigkeit"}},
            ],
        ))

    fig.update_layout(
        title=f"Häufigkeit der Wörter {search_terms}",
        xaxis_title="Zeit",
        yaxis_title="Absolute Häufigkeit",
        width=800,
        height=450,
        updatemenus=[dict(
            buttons=buttons,
            direction="down",
            x=1.02, xanchor="left",
            y=1.0,  yanchor="top",
            showactive=True,
        )],
    )
    fig.show()

In [ ]:
# Call the function to plot the frequencies 
plot_frequencies(corpus_annotations_merged, search_terms=word_list)

### Worteingabe für die Suche (für Cloud Mode und Local Mode) 

In [ ]:
text_input = input("Geben Sie die zu suchenden Wörter ein und trennen Sie sie durch Kommas, wenn es mehrere sind:")
# Convert the input to a list by splitting the input by comma
text_input = [x.strip() for x in text_input.split(',')]

In [ ]:
plot_frequencies(corpus_annotations_merged, search_terms=text_input)

## Diskussion des Zwischenergebnisses

Ist dieses Ergebnis sinnvoll und spiegelt es tatsächlich etwas wider? Eine Möglichkeit, dies zu überprüfen, besteht darin, unser Diagramm mit den tatsächlichen Daten über die Intensität der Pandemie zu vergleichen.

In Taubenberger & Morens (2006) wird festgestellt, dass 'The first pandemic influenza wave appeared in the spring of 1918, followed in rapid succession by much more fatal second and third waves in the fall and winter of 1918–1919, respectively' ('Die erste pandemische Influenza-Welle im Frühjahr 1918 auftrat, gefolgt von weitaus tödlicheren zweiten und dritten Wellen im Herbst und Winter 1918–1919'). Sie ergänzen diese Aussage auch mit einem Diagramm aus einem früheren Papier (Jordan 1927):

<img src="https://wwwnc.cdc.gov/eid/images/05-0979-F1.gif">

Unsere zwei Wellen der Erwähnungen des Wortes 'Grippe' scheinen den Sterblichkeitszahlen zu entsprechen, was darauf hindeuten könnte, dass die Methode, obwohl sehr einfach, funktioniert und dass historische Ereignisse manchmal in Wortfrequenzzählungen reflektiert werden können... Die dritte Welle scheint nicht reproduziert zu werden, was eine weitere Untersuchung erfordert. Eine Hypothese könnte sein, dass, ähnlich wie bei der COVID-Pandemie, neue Krankheitswellen irgendwann aufhören, die Aufmerksamkeit der Öffentlichkeit zu erregen. Beispielsweise waren die COVID-Wellen im Jahr 2021 stärker als die im Jahr 2020, aber die Berichterstattung in den Nachrichten nahm bereits ab. Dies könnte besonders für Anfang 1919 zutreffen, als nach dem Verlust des Krieges und der Revolution von 1918 Grippetodesfälle kein Nachrichtenthema mehr waren.

### Bibliographie
* Jordan E. (1927). Epidemic influenza: a survey. Chicago: American Medical Association.
* Taubenberger, J. K., & Morens, D. M. (2006). 1918 Influenza: the Mother of All Pandemics. Emerging Infectious Diseases, 12(1), 15-22. https://doi.org/10.3201/eid1201.050979